# Assessment 2 - Financial Accounting & GL Reconciliation

Finance reports that balances generated from the new data platform do not reconcile with the
bank's General Ledger. This notebook traces the cause across four tasks: GL arithmetic integrity
and reconciliation, accounting mapping validation, a structured variance investigation, and a
reconciliation framework design. See `results/assessment-2/assessment-2-overview.md` for the full
scenario, table shapes, and scale. Connectivity conventions: see `00_template_connectivity_check.ipynb`.


In [1]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]


## Task 1 - Validate Accounting Integrity

Confirm `opening_balance + debit_movement - credit_movement = closing_balance` on the General
Ledger, identify violations, then independently recompute expected debit/credit movements from the
transaction-level data and reconcile against the General Ledger at legal entity, GL account, cost
center, currency, and accounting date.


In [2]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-task1-gl-integrity")
    .getOrCreate()
)


def jdbc_table(table_name):
    return spark.read.jdbc(
        url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
        table=table_name,
        properties={
            "user": POSTGRES_USER,
            "password": POSTGRES_PASSWORD,
            "driver": "org.postgresql.Driver",
        },
    )


gl_df = jdbc_table("finance.gl_balance")
txn_df = jdbc_table("bronze.finance_transactions")
map_df = jdbc_table("ref.accounting_mapping")

txn_df.createOrReplaceTempView("finance_transactions")
map_df.createOrReplaceTempView("accounting_mapping")

print(f"[INFO] finance.gl_balance row_count={gl_df.count()}")
print(f"[INFO] bronze.finance_transactions row_count={txn_df.count()}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 05:59:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[INFO] finance.gl_balance row_count=589


[INFO] bronze.finance_transactions row_count=1523


### Arithmetic integrity

`opening_balance + debit_movement - credit_movement = closing_balance` checked on every General
Ledger row. Tolerance: exact equality - all four columns are `decimal(20,2)` and the expression is
pure addition/subtraction, so a nonzero result is a genuine arithmetic break, not a rounding artifact.


In [3]:
from pyspark.sql.functions import col, sum as spark_sum, abs as spark_abs

# kept as native decimal(20,2) here (no cast to double) - exact-equality comparisons need to stay in
# decimal arithmetic; a double cast introduces float rounding noise well below the cent level, which
# would silently inflate this count. Double is used further down only where a numeric tolerance
# (e.g. one cent) already absorbs that noise.
gl_exact = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"),
    col("cost_center"), col("currency"),
    col("opening_balance"), col("debit_movement"), col("credit_movement"), col("closing_balance"),
)

arithmetic_check = gl_exact.withColumn(
    "computed_closing", col("opening_balance") + col("debit_movement") - col("credit_movement")
).withColumn(
    "variance", col("closing_balance") - col("computed_closing")
)

arithmetic_violations = arithmetic_check.filter(col("variance") != 0)
violation_count = arithmetic_violations.count()

print(f"[INFO] arithmetic integrity violations={violation_count}")
arithmetic_violations.select(
    "accounting_date", "legal_entity", "gl_account", "cost_center", "currency",
    "closing_balance", "computed_closing", "variance",
).show(20, truncate=False)


[INFO] arithmetic integrity violations=5


+---------------+------------+----------+-----------+--------+---------------+----------------+--------+
|accounting_date|legal_entity|gl_account|cost_center|currency|closing_balance|computed_closing|variance|
+---------------+------------+----------+-----------+--------+---------------+----------------+--------+
|2026-08-17     |LE3         |GL9999    |CC99       |SGD     |-146704.97     |-147512.45      |807.48  |
|2026-08-18     |LE4         |GL1009    |CC10       |EUR     |51299.80       |48491.98        |2807.82 |
|2026-08-20     |LE1         |GL1002    |CC03       |USD     |19036.91       |16431.98        |2604.93 |
|2026-08-20     |LE2         |GL1009    |CC10       |USD     |92059.31       |93219.61        |-1160.30|
|2026-08-21     |LE3         |GL1009    |CC10       |SGD     |205175.48      |200560.78       |4614.70 |
+---------------+------------+----------+-----------+--------+---------------+----------------+--------+



### Independent movement recomputation

Recomputed from `bronze.finance_transactions` on the transaction's *expected* classification, not
its actual (as-posted) one - grouping by the same values the Ledger was built from would make this
check tautological, since the Ledger is itself built from those actual, possibly-misclassified
values. `expected_gl_account`/`expected_cost_center` come from `ref.accounting_mapping` wherever a
transaction matches exactly one active mapping row; `legal_entity` uses the majority-vote value for
the transaction's account (the same heuristic used later for the incorrect-legal-entity check),
since the mapping table carries no legal-entity field. Six product/transaction-type combinations
have more than one currently-active, *conflicting* mapping row (Task 2's own overlapping/multi-GL
findings) - there is no single unambiguous expected value for these, so they fall back to the
transaction's actual classification rather than an arbitrary pick among equally-valid candidates.

The mapping lookup itself is keyed on the transaction's own posted debit/credit indicator, so a
transaction carrying the wrong indicator (see Task 3's own check on this) would otherwise be looked
up under the wrong key - matched against the mapping row for the *opposite* of its true type, and
so measured against the wrong expected value. A transaction whose actual classification doesn't
match any active mapping row under its own indicator, but exactly matches one under the opposite
indicator, is looked up under that opposite indicator instead. `local_amount` is the SGD amount
basis for the aggregate variance and is compared with the Ledger's `local_sgd_debit_movement` and
`local_sgd_credit_movement` fields. Native `transaction_amount` remains useful only for diagnostics
that keep `currency` in the grouping and do not collapse currencies into one monetary total.

Tolerance: `0.01` SGD applied independently to each side of the movement. A full outer join (not
left/inner) means a Ledger key with no matching transactions, or a transaction key with no matching
Ledger row, both surface as a variance instead of silently dropping out.


In [4]:
from pyspark.sql.functions import when, coalesce

MOVEMENT_TOLERANCE_ABS = 0.01

gl = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"),
    col("cost_center"), col("currency"),
    col("local_sgd_debit_movement").cast("double").alias("debit_movement"),
    col("local_sgd_credit_movement").cast("double").alias("credit_movement"),
)

# exactly one mapping row per transaction, dropping the 6 combos with more than one active,
# conflicting mapping row (no single unambiguous "expected" value exists for those).

# a transaction whose actual (gl_account, cost_center) doesn't match any currently-active mapping
# row for its own posted indicator, but exactly matches one for the opposite indicator, is treated
# as carrying the wrong debit/credit indicator - its own values are internally consistent with a
# real, valid combination, just filed under the wrong sign.
flip_candidates = spark.sql('''
    WITH current_match AS (
      SELECT t.transaction_id,
             MAX(CASE WHEN t.gl_account = m.expected_gl_account AND t.cost_center = m.expected_cost_center THEN 1 ELSE 0 END) AS matches_current
      FROM finance_transactions t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
        AND t.transaction_date >= m.effective_start_date AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
      GROUP BY t.transaction_id
    ),
    opposite_match AS (
      SELECT t.transaction_id,
             MAX(CASE WHEN t.gl_account = m.expected_gl_account AND t.cost_center = m.expected_cost_center THEN 1 ELSE 0 END) AS matches_opposite
      FROM finance_transactions t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code
        AND m.transaction_type = CASE WHEN t.debit_credit_indicator = 'DEBIT' THEN 'CREDIT' ELSE 'DEBIT' END
        AND t.transaction_date >= m.effective_start_date AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
      GROUP BY t.transaction_id
    )
    SELECT t.transaction_id
    FROM finance_transactions t
    LEFT JOIN current_match cm ON t.transaction_id = cm.transaction_id
    JOIN opposite_match om ON t.transaction_id = om.transaction_id
    WHERE COALESCE(cm.matches_current, 0) = 0 AND om.matches_opposite = 1
''')
flip_candidates.createOrReplaceTempView("flip_candidates")

# the mapping lookup itself is keyed on the transaction's indicator - a flip candidate's own
# posted indicator is the wrong key to look up under, so the opposite indicator is used instead.
canonical_mapping = spark.sql('''
    WITH corrected AS (
      SELECT t.*, CASE WHEN fc.transaction_id IS NOT NULL
                        THEN (CASE WHEN t.debit_credit_indicator = 'DEBIT' THEN 'CREDIT' ELSE 'DEBIT' END)
                        ELSE t.debit_credit_indicator END AS lookup_type
      FROM finance_transactions t
      LEFT JOIN flip_candidates fc ON t.transaction_id = fc.transaction_id
    ),
    matched AS (
      SELECT t.transaction_id, m.expected_gl_account, m.expected_cost_center,
             COUNT(*) OVER (PARTITION BY t.transaction_id) AS match_count
      FROM corrected t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code AND t.lookup_type = m.transaction_type
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    )
    SELECT DISTINCT transaction_id, expected_gl_account, expected_cost_center
    FROM matched WHERE match_count = 1
''')

# an account's most-frequently-posted legal entity elsewhere in the period - used as the "expected"
# value for this dimension since the mapping table carries no legal-entity field.
account_entity_mode = spark.sql('''
    SELECT account_id, legal_entity FROM (
      SELECT account_id, legal_entity,
             ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY COUNT(*) DESC) AS rnk
      FROM finance_transactions GROUP BY account_id, legal_entity
    ) WHERE rnk = 1
''')

txn = txn_df.alias("t").join(
    canonical_mapping.alias("cm"), col("t.transaction_id") == col("cm.transaction_id"), "left"
).join(
    account_entity_mode.alias("em"), col("t.account_id") == col("em.account_id"), "left"
).select(
    col("t.posting_date").alias("accounting_date"),
    coalesce(col("em.legal_entity"), col("t.legal_entity")).alias("legal_entity"),
    coalesce(col("cm.expected_gl_account"), col("t.gl_account")).alias("gl_account"),
    coalesce(col("cm.expected_cost_center"), col("t.cost_center")).alias("cost_center"),
    col("t.currency"), col("t.debit_credit_indicator"),
    col("t.local_amount").cast("double").alias("recompute_amount"),
)

recomputed = txn.groupBy("accounting_date", "legal_entity", "gl_account", "cost_center", "currency").agg(
    spark_sum(when(col("debit_credit_indicator") == "DEBIT", col("recompute_amount")).otherwise(0.0)).alias("recomputed_debit"),
    spark_sum(when(col("debit_credit_indicator") == "CREDIT", col("recompute_amount")).otherwise(0.0)).alias("recomputed_credit"),
)

JOIN_KEYS = ["accounting_date", "legal_entity", "gl_account", "cost_center", "currency"]

movement_compare = gl.join(recomputed, JOIN_KEYS, "full_outer").select(
    *[col(k) for k in JOIN_KEYS],
    col("debit_movement"), col("recomputed_debit"),
    col("credit_movement"), col("recomputed_credit"),
).fillna(0.0, subset=["debit_movement", "recomputed_debit", "credit_movement", "recomputed_credit"]).withColumn(
    "debit_variance", col("debit_movement") - col("recomputed_debit")
).withColumn(
    "credit_variance", col("credit_movement") - col("recomputed_credit")
)

movement_variances = movement_compare.filter(
    (spark_abs(col("debit_variance")) > MOVEMENT_TOLERANCE_ABS)
    | (spark_abs(col("credit_variance")) > MOVEMENT_TOLERANCE_ABS)
)
movement_compare.cache()
movement_variance_count = movement_variances.count()
movement_variance_total = movement_variances.agg(
    spark_sum(spark_abs(col("debit_variance")) + spark_abs(col("credit_variance")))
).collect()[0][0] or 0.0

print(f"[INFO] SGD movement recomputation variances={movement_variance_count} total={movement_variance_total:.2f}")
movement_variances.orderBy(spark_abs(col("debit_variance") + col("credit_variance")).desc()).show(20, truncate=False)


[INFO] SGD movement recomputation variances=30 total=297137.40


+---------------+------------+----------+-----------+--------+--------------+------------------+---------------+------------------+------------------+-------------------+
|accounting_date|legal_entity|gl_account|cost_center|currency|debit_movement|recomputed_debit  |credit_movement|recomputed_credit |debit_variance    |credit_variance    |
+---------------+------------+----------+-----------+--------+--------------+------------------+---------------+------------------+------------------+-------------------+
|2026-08-21     |LE2         |GL1000    |CC01       |USD     |0.0           |22174.44          |0.0            |0.0               |-22174.44         |0.0                |
|2026-08-21     |LE2         |GL1000    |CC02       |USD     |22174.44      |0.0               |0.0            |0.0               |22174.44          |0.0                |
|2026-08-21     |LE1         |GL9999    |CC99       |SGD     |17639.6       |17639.6           |125944.41      |107478.93000000001|0.0           

### Dimensional reconciliation

The same recomputation, rolled up to one dimension at a time instead of the full five-key grain.
Reconciliation status per row: `PASS` if `variance_pct < 0.1%`, `WARNING` if `< 1%`, `FAIL` otherwise.


In [5]:
def status_for(variance_pct):
    pct = abs(variance_pct)
    if pct < 0.1:
        return "PASS"
    if pct < 1.0:
        return "WARNING"
    return "FAIL"


LEVEL_DIMENSIONS = ["legal_entity", "gl_account", "cost_center", "currency", "accounting_date"]

dimension_summaries = {}

for dim in LEVEL_DIMENSIONS:
    rolled = movement_compare.groupBy(dim).agg(
        spark_sum("debit_movement").alias("gl_debit"),
        spark_sum("recomputed_debit").alias("recomputed_debit"),
        spark_sum("credit_movement").alias("gl_credit"),
        spark_sum("recomputed_credit").alias("recomputed_credit"),
    ).withColumn(
        "debit_variance", col("gl_debit") - col("recomputed_debit")
    ).withColumn(
        "credit_variance", col("gl_credit") - col("recomputed_credit")
    ).collect()

    rows = []
    for r in rolled:
        gl_total = (r["gl_debit"] or 0.0) + (r["gl_credit"] or 0.0)
        var_total = (r["debit_variance"] or 0.0) + (r["credit_variance"] or 0.0)
        variance_pct = round((abs(var_total) / gl_total * 100) if gl_total else 0.0, 4)
        rows.append({
            "group_value": r[dim], "gl_debit": r["gl_debit"], "gl_credit": r["gl_credit"],
            "debit_variance": r["debit_variance"], "credit_variance": r["credit_variance"],
            "variance_pct": variance_pct, "status": status_for(variance_pct),
        })
    dimension_summaries[dim] = rows
    worst = sorted(rows, key=lambda r: abs(r["variance_pct"]), reverse=True)[:5]
    print(f"[INFO] dimensional reconciliation by {dim} - top variance groups:")
    for r in worst:
        print(f"  [{r['status']}] {dim}={r['group_value']} debit_variance={r['debit_variance']:.2f} "
              f"credit_variance={r['credit_variance']:.2f} variance_pct={r['variance_pct']}%")


[INFO] dimensional reconciliation by legal_entity - top variance groups:
  [WARNING] legal_entity=LE1 debit_variance=0.00 credit_variance=35228.24 variance_pct=0.7482%
  [WARNING] legal_entity=LE4 debit_variance=0.00 credit_variance=-25229.20 variance_pct=0.6179%
  [WARNING] legal_entity=LE3 debit_variance=0.00 credit_variance=-6094.13 variance_pct=0.137%
  [WARNING] legal_entity=LE2 debit_variance=0.00 credit_variance=-3904.91 variance_pct=0.1039%


[INFO] dimensional reconciliation by gl_account - top variance groups:
  [FAIL] gl_account=GL1005 debit_variance=-0.00 credit_variance=33646.03 variance_pct=4.6907%
  [FAIL] gl_account=GL1012 debit_variance=0.00 credit_variance=-33646.03 variance_pct=4.6818%
  [FAIL] gl_account=GL1011 debit_variance=-29748.14 credit_variance=0.00 variance_pct=3.1714%
  [FAIL] gl_account=GL1007 debit_variance=20041.41 credit_variance=0.00 variance_pct=1.9602%
  [WARNING] gl_account=GL9999 debit_variance=9706.73 credit_variance=-0.00 variance_pct=0.2232%


[INFO] dimensional reconciliation by cost_center - top variance groups:
  [FAIL] cost_center=CC09 debit_variance=16506.33 credit_variance=0.00 variance_pct=2.2957%
  [FAIL] cost_center=CC01 debit_variance=-22174.44 credit_variance=0.00 variance_pct=1.267%
  [WARNING] cost_center=CC99 debit_variance=9706.73 credit_variance=-0.00 variance_pct=0.2247%
  [WARNING] cost_center=CC07 debit_variance=0.00 credit_variance=-1355.23 variance_pct=0.1756%
  [WARNING] cost_center=CC02 debit_variance=-4038.62 credit_variance=1355.23 variance_pct=0.1523%


[INFO] dimensional reconciliation by currency - top variance groups:
  [PASS] currency=EUR debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%
  [PASS] currency=USD debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%
  [PASS] currency=SGD debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%


[INFO] dimensional reconciliation by accounting_date - top variance groups:
  [PASS] accounting_date=2026-08-18 debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%
  [PASS] accounting_date=2026-08-19 debit_variance=0.00 credit_variance=-0.00 variance_pct=0.0%
  [PASS] accounting_date=2026-08-17 debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%
  [PASS] accounting_date=2026-08-21 debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%
  [PASS] accounting_date=2026-08-20 debit_variance=-0.00 credit_variance=0.00 variance_pct=0.0%


### Reconciliation framework write-back

The reconciliation control table's `dimension` column is a closed set (`row_count`, `amount`); the
fine-grained per-dimension detail above is reported in this notebook and the reconciliation-results
deliverable only. `row_count` compares total transaction count against total Ledger row count.
`amount` compares source SGD transaction value (`local_amount`) against GL SGD movement value
(`local_sgd_debit_movement + local_sgd_credit_movement`) - a common-currency movement total. Native
amounts remain available for per-currency diagnostics, but they are not collapsed into a single
monetary aggregate.


In [6]:
def reserve_batch_id(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT nextval('reconciliation.rc_batch_control_batch_id_seq');")
        return cur.fetchone()[0]


def insert_batch_control(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_batch_control (batch_id, batch_date, assessment_id, status) "
            "VALUES (%s, CURRENT_DATE, %s, %s)",
            (batch_id, "assessment-2", status),
        )


def update_batch_status(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "UPDATE reconciliation.rc_batch_control SET status = %s WHERE batch_id = %s",
            (status, batch_id),
        )


def insert_result_row(conn, batch_id, dimension, source_value, target_value):
    variance = target_value - source_value
    variance_pct = round((variance / source_value * 100) if source_value else 0.0, 4)
    status = status_for(variance_pct)
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_reconciliation_results "
            "(batch_id, dimension, source_value, target_value, variance, variance_pct, reconciliation_status) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            (batch_id, dimension, source_value, target_value, variance, variance_pct, status),
        )
    return status


txn_count = txn_df.count()
gl_count = gl_df.count()
txn_amount = txn_df.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
gl_amount = gl_df.agg(
    spark_sum(col("local_sgd_debit_movement").cast("double"))
    + spark_sum(col("local_sgd_credit_movement").cast("double"))
).collect()[0][0] or 0.0

conn = psycopg2.connect(host="postgres", port=5432, dbname=POSTGRES_DB, user=POSTGRES_USER, password=POSTGRES_PASSWORD)
batch_id = reserve_batch_id(conn)
insert_batch_control(conn, batch_id, "RUNNING")

statuses = [
    insert_result_row(conn, batch_id, "row_count", txn_count, gl_count),
    insert_result_row(conn, batch_id, "amount", txn_amount, gl_amount),
]
overall_status = max(statuses, key=lambda s: {"PASS": 0, "WARNING": 1, "FAIL": 2}[s])
update_batch_status(conn, batch_id, overall_status)
conn.commit()
conn.close()

print(f"[INFO] source(transactions): count={txn_count} SGD amount={txn_amount:.2f}")
print(f"[INFO] target(general ledger): count={gl_count} SGD amount={gl_amount:.2f}")
print(f"[{overall_status}] task 1 batch reconciliation: batch_id={batch_id}")


[INFO] source(transactions): count=1523 SGD amount=16999151.01
[INFO] target(general ledger): count=589 SGD amount=16999151.01
[FAIL] task 1 batch reconciliation: batch_id=26


In [7]:
spark.stop()


## Task 2 - Validate Accounting Mapping

Using the accounting mapping table: determine whether transactions are posted to the expected GL
accounts, validate mapping effective dates, identify transactions with missing accounting mappings,
detect overlapping effective-date mappings, identify expired mappings still being used, and identify
products mapped to multiple GL accounts unexpectedly.

Implemented below via Spark SQL over temp views rather than the DataFrame API - the effective-dated
join and the self-joins for overlap/multi-GL detection read directly as SQL.


In [8]:
from pyspark.sql.functions import lit

spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-task2-mapping-validation")
    .getOrCreate()
)

txn_df = jdbc_table("bronze.finance_transactions")
map_df = jdbc_table("ref.accounting_mapping")

txn_df.createOrReplaceTempView("finance_transactions")
map_df.createOrReplaceTempView("accounting_mapping")

print(f"[INFO] bronze.finance_transactions row_count={txn_df.count()}")
print(f"[INFO] ref.accounting_mapping row_count={map_df.count()}")


[INFO] bronze.finance_transactions row_count=1523


[INFO] ref.accounting_mapping row_count=22


### Join key

The mapping table's transaction type and the transaction table's debit/credit indicator carry the
same domain (`DEBIT`/`CREDIT`) under different column names; every check below effective-dates the
join on the transaction's own `transaction_date` (not `posting_date` - the mapping rule governs which
policy applied when the transaction occurred, independent of when it was later posted).


### Posted to expected GL account

If the join returns more than one mapping row per transaction (an overlapping-range case), every
matched row is evaluated independently rather than one being picked arbitrarily - a transaction is
flagged if it disagrees with *any* matched mapping.


In [9]:
gl_mismatch = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           m.expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON  t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
      AND t.transaction_date >= m.effective_start_date
      AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    WHERE t.gl_account <> m.expected_gl_account
''').withColumn("exception", lit("GL_MISMATCH"))

gl_mismatch_rows = gl_mismatch.count()
gl_mismatch_txns = gl_mismatch.select("transaction_id").distinct().count()
print(f"[INFO] posted to expected GL account - GL_MISMATCH rows={gl_mismatch_rows} distinct_transactions={gl_mismatch_txns}")
gl_mismatch.show(10, truncate=False)


[INFO] posted to expected GL account - GL_MISMATCH rows=403 distinct_transactions=319


+--------------+------------+---------+-------------------+---------------+-----------+
|transaction_id|product_code|actual_gl|expected_gl_account|accounting_date|exception  |
+--------------+------------+---------+-------------------+---------------+-----------+
|FTX-0000008   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000012   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000020   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000023   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000027   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000042   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000080   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000100   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000114   |P1          |GL1

### Mapping effective-date validity and missing mapping

A mapping-effective-date violation is a `(product_code, debit_credit_indicator)` pair that exists in
the mapping table but for which no row's window covers `transaction_date`. A missing mapping is
distinguished from that by whether *any* row exists for the pair at all, not just whether one covers
the right date.


In [10]:
no_effective_mapping = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           CAST(NULL AS STRING) AS expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    WHERE EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
    )
    AND NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    )
''').withColumn("exception", lit("NO_EFFECTIVE_MAPPING"))

mapping_not_found = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           CAST(NULL AS STRING) AS expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
    )
''').withColumn("exception", lit("MAPPING_NOT_FOUND"))

print(f"[INFO] mapping effective-date validity - NO_EFFECTIVE_MAPPING rows={no_effective_mapping.count()}")
print(f"[INFO] missing accounting mapping - MAPPING_NOT_FOUND rows={mapping_not_found.count()}")
mapping_not_found.show(5, truncate=False)


[INFO] mapping effective-date validity - NO_EFFECTIVE_MAPPING rows=0


[INFO] missing accounting mapping - MAPPING_NOT_FOUND rows=385


+--------------+------------+---------+-------------------+---------------+-----------------+
|transaction_id|product_code|actual_gl|expected_gl_account|accounting_date|exception        |
+--------------+------------+---------+-------------------+---------------+-----------------+
|FTX-0000019   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000026   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000028   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000043   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000051   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
+--------------+------------+---------+-------------------+---------------+-----------------+
only showing top 5 rows



### Overlapping effective-date mapping ranges

Mapping-level, not transaction-level: two rows for the same `(product_code, transaction_type)` whose
windows intersect.


In [11]:
overlapping_mapping = spark.sql('''
    SELECT a.product_code, a.transaction_type, a.effective_start_date, a.effective_end_date,
           b.effective_start_date AS overlap_start, b.effective_end_date AS overlap_end
    FROM accounting_mapping a
    JOIN accounting_mapping b
      ON  a.product_code = b.product_code AND a.transaction_type = b.transaction_type
      AND a.effective_start_date < b.effective_start_date
      AND a.effective_start_date <= COALESCE(b.effective_end_date, DATE '9999-12-31')
      AND COALESCE(a.effective_end_date, DATE '9999-12-31') >= b.effective_start_date
''')

overlapping_mapping_count = overlapping_mapping.count()
print(f"[INFO] overlapping effective-date mapping ranges - pairs={overlapping_mapping_count}")
overlapping_mapping.show(10, truncate=False)


[INFO] overlapping effective-date mapping ranges - pairs=8


+------------+----------------+--------------------+------------------+-------------+-----------+
|product_code|transaction_type|effective_start_date|effective_end_date|overlap_start|overlap_end|
+------------+----------------+--------------------+------------------+-------------+-----------+
|P1          |CREDIT          |2025-08-17          |NULL              |2026-07-18   |NULL       |
|P1          |DEBIT           |2025-08-17          |NULL              |2026-07-18   |NULL       |
|P1          |DEBIT           |2025-08-17          |NULL              |2026-08-12   |NULL       |
|P1          |DEBIT           |2026-07-18          |NULL              |2026-08-12   |NULL       |
|P4          |DEBIT           |2025-08-17          |NULL              |2026-01-29   |2026-08-07 |
|P5          |DEBIT           |2025-08-17          |NULL              |2026-08-12   |NULL       |
|P8          |CREDIT          |2025-08-17          |NULL              |2026-01-29   |2026-08-07 |
|P9          |CREDIT

### Expired mapping still referenced

A transaction can reference an expired mapping row while *also* having a currently-valid mapping row
it should have used instead - that combination is a GL mismatch, not an expired-mapping case. This
fires only when the expired row is the sole candidate.


In [12]:
expired_mapping = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           m.expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
    WHERE m.effective_end_date IS NOT NULL
      AND t.transaction_date > m.effective_end_date
      AND NOT EXISTS (
        SELECT 1 FROM accounting_mapping m2
        WHERE m2.product_code = t.product_code AND m2.transaction_type = t.debit_credit_indicator
          AND t.transaction_date >= m2.effective_start_date
          AND (t.transaction_date <= m2.effective_end_date OR m2.effective_end_date IS NULL)
      )
''').withColumn("exception", lit("EXPIRED_MAPPING"))

expired_mapping_rows = expired_mapping.count()
print(f"[INFO] expired mapping still referenced rows={expired_mapping_rows}")
expired_mapping.show(10, truncate=False)


[INFO] expired mapping still referenced rows=0


+--------------+------------+---------+-------------------+---------------+---------+
|transaction_id|product_code|actual_gl|expected_gl_account|accounting_date|exception|
+--------------+------------+---------+-------------------+---------------+---------+
+--------------+------------+---------+-------------------+---------------+---------+



### Product mapped to multiple GL accounts unexpectedly

Mapping-level: currently-active rows (open-ended or not yet expired) that disagree on the expected GL
account - a genuine data conflict rather than a time-ordered supersession.


In [13]:
multi_gl_mapping = spark.sql('''
    SELECT product_code, transaction_type, COUNT(DISTINCT expected_gl_account) AS gl_account_count
    FROM accounting_mapping
    WHERE effective_end_date IS NULL OR effective_end_date >= CURRENT_DATE
    GROUP BY product_code, transaction_type
    HAVING COUNT(DISTINCT expected_gl_account) > 1
''')

multi_gl_mapping_count = multi_gl_mapping.count()
print(f"[INFO] product mapped to multiple GL accounts - product/type pairs={multi_gl_mapping_count}")
multi_gl_mapping.show(10, truncate=False)


[INFO] product mapped to multiple GL accounts - product/type pairs=4


+------------+----------------+----------------+
|product_code|transaction_type|gl_account_count|
+------------+----------------+----------------+
|P1          |DEBIT           |3               |
|P9          |CREDIT          |2               |
|P1          |CREDIT          |2               |
|P5          |DEBIT           |2               |
+------------+----------------+----------------+



### Exception output

The assignment's own shape - `Transaction, Product, Actual GL, Expected GL, Accounting Date,
Exception` - applies to the four per-transaction checks above; the two mapping-level findings (no
`transaction_id` to key on) are reported separately.


In [14]:
mapping_exceptions = (
    gl_mismatch
    .unionByName(no_effective_mapping)
    .unionByName(mapping_not_found)
    .unionByName(expired_mapping)
)

exception_output = mapping_exceptions.select(
    col("transaction_id").alias("Transaction"),
    col("product_code").alias("Product"),
    col("actual_gl").alias("Actual GL"),
    col("expected_gl_account").alias("Expected GL"),
    col("accounting_date").alias("Accounting Date"),
    col("exception").alias("Exception"),
)

total_exceptions = exception_output.count()
print(f"[INFO] task 2 exception output rows={total_exceptions}")
exception_output.groupBy("Exception").count().orderBy(col("count").desc()).show(truncate=False)


[INFO] task 2 exception output rows=788


+-----------------+-----+
|Exception        |count|
+-----------------+-----+
|GL_MISMATCH      |403  |
|MAPPING_NOT_FOUND|385  |
+-----------------+-----+



In [15]:
print(f"[PASS] task 2 accounting mapping validation: exception_rows={total_exceptions}")
spark.stop()


[PASS] task 2 accounting mapping validation: exception_rows=788


## Task 3 - Investigate a Finance Variance

Finance reports a material variance between the expected and platform closing balances. The dataset
may contain any combination of duplicate accounting entries, transactions posted twice, incorrect
debit/credit indicators, incorrect FX conversion, missing accounting mappings, transactions posted
one accounting day late, incorrect legal-entity allocation, and incorrect cost-center assignment.
Each is checked independently below and compared against the variance Task 1's expected-classification
recomputation found.


In [16]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-task3-variance-investigation")
    .getOrCreate()
)

txn_df = jdbc_table("bronze.finance_transactions")
map_df = jdbc_table("ref.accounting_mapping")
gl_df = jdbc_table("finance.gl_balance")
txn_df.createOrReplaceTempView("finance_transactions")
map_df.createOrReplaceTempView("accounting_mapping")

# task 1's movement_compare is a DataFrame tied to task 1's (now-stopped) SparkSession - rebuilding
# it here, identically, against this section's own session, rather than reusing the dead reference.
gl = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"),
    col("cost_center"), col("currency"),
    col("local_sgd_debit_movement").cast("double").alias("debit_movement"),
    col("local_sgd_credit_movement").cast("double").alias("credit_movement"),
)

# a transaction whose actual (gl_account, cost_center) doesn't match any currently-active mapping
# row for its own posted indicator, but exactly matches one for the opposite indicator, is treated
# as carrying the wrong debit/credit indicator - its own values are internally consistent with a
# real, valid combination, just filed under the wrong sign.
flip_candidates = spark.sql('''
    WITH current_match AS (
      SELECT t.transaction_id,
             MAX(CASE WHEN t.gl_account = m.expected_gl_account AND t.cost_center = m.expected_cost_center THEN 1 ELSE 0 END) AS matches_current
      FROM finance_transactions t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
        AND t.transaction_date >= m.effective_start_date AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
      GROUP BY t.transaction_id
    ),
    opposite_match AS (
      SELECT t.transaction_id,
             MAX(CASE WHEN t.gl_account = m.expected_gl_account AND t.cost_center = m.expected_cost_center THEN 1 ELSE 0 END) AS matches_opposite
      FROM finance_transactions t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code
        AND m.transaction_type = CASE WHEN t.debit_credit_indicator = 'DEBIT' THEN 'CREDIT' ELSE 'DEBIT' END
        AND t.transaction_date >= m.effective_start_date AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
      GROUP BY t.transaction_id
    )
    SELECT t.transaction_id
    FROM finance_transactions t
    LEFT JOIN current_match cm ON t.transaction_id = cm.transaction_id
    JOIN opposite_match om ON t.transaction_id = om.transaction_id
    WHERE COALESCE(cm.matches_current, 0) = 0 AND om.matches_opposite = 1
''')
flip_candidates.createOrReplaceTempView("flip_candidates")

# the mapping lookup itself is keyed on the transaction's indicator - a flip candidate's own
# posted indicator is the wrong key to look up under, so the opposite indicator is used instead.
canonical_mapping = spark.sql('''
    WITH corrected AS (
      SELECT t.*, CASE WHEN fc.transaction_id IS NOT NULL
                        THEN (CASE WHEN t.debit_credit_indicator = 'DEBIT' THEN 'CREDIT' ELSE 'DEBIT' END)
                        ELSE t.debit_credit_indicator END AS lookup_type
      FROM finance_transactions t
      LEFT JOIN flip_candidates fc ON t.transaction_id = fc.transaction_id
    ),
    matched AS (
      SELECT t.transaction_id, m.expected_gl_account, m.expected_cost_center,
             COUNT(*) OVER (PARTITION BY t.transaction_id) AS match_count
      FROM corrected t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code AND t.lookup_type = m.transaction_type
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    )
    SELECT DISTINCT transaction_id, expected_gl_account, expected_cost_center
    FROM matched WHERE match_count = 1
''')
canonical_mapping.createOrReplaceTempView("canonical_mapping")

account_entity_mode_t1 = spark.sql('''
    SELECT account_id, legal_entity FROM (
      SELECT account_id, legal_entity,
             ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY COUNT(*) DESC) AS rnk
      FROM finance_transactions GROUP BY account_id, legal_entity
    ) WHERE rnk = 1
''')
txn_expected = txn_df.alias("t").join(
    canonical_mapping.alias("cm"), col("t.transaction_id") == col("cm.transaction_id"), "left"
).join(
    account_entity_mode_t1.alias("em"), col("t.account_id") == col("em.account_id"), "left"
).select(
    col("t.posting_date").alias("accounting_date"),
    coalesce(col("em.legal_entity"), col("t.legal_entity")).alias("legal_entity"),
    coalesce(col("cm.expected_gl_account"), col("t.gl_account")).alias("gl_account"),
    coalesce(col("cm.expected_cost_center"), col("t.cost_center")).alias("cost_center"),
    col("t.currency"), col("t.debit_credit_indicator"),
    col("t.local_amount").cast("double").alias("recompute_amount"),
)
recomputed_t1 = txn_expected.groupBy("accounting_date", "legal_entity", "gl_account", "cost_center", "currency").agg(
    spark_sum(when(col("debit_credit_indicator") == "DEBIT", col("recompute_amount")).otherwise(0.0)).alias("recomputed_debit"),
    spark_sum(when(col("debit_credit_indicator") == "CREDIT", col("recompute_amount")).otherwise(0.0)).alias("recomputed_credit"),
)
JOIN_KEYS = ["accounting_date", "legal_entity", "gl_account", "cost_center", "currency"]
movement_compare = gl.join(recomputed_t1, JOIN_KEYS, "full_outer").fillna(
    0.0, subset=["debit_movement", "recomputed_debit", "credit_movement", "recomputed_credit"]
).withColumn("debit_variance", col("debit_movement") - col("recomputed_debit")
).withColumn("credit_variance", col("credit_movement") - col("recomputed_credit"))
movement_compare.cache()
movement_variance_total = movement_compare.filter(
    (spark_abs(col("debit_variance")) > 0.01) | (spark_abs(col("credit_variance")) > 0.01)
).agg(spark_sum(spark_abs(col("debit_variance")) + spark_abs(col("credit_variance")))).collect()[0][0] or 0.0
print(f"[INFO] task 1 SGD movement variance restated in this session: {movement_variance_total:.2f}")


[INFO] task 1 SGD movement variance restated in this session: 297137.40


### Duplicate accounting entries and transactions posted twice

Hash the business fields (every column except `transaction_id`) and find hash collisions across
distinct transaction ids posted on the same posting date - every row in a matching group is a
candidate duplicate. This single detection surfaces both assignment-named scenarios equally: a
straight duplicate entry and a transaction re-posted under a different id are indistinguishable from
the data alone, since both manifest as more than one transaction id carrying identical business-field
content on the same posting date. Not expected to move Task 1's variance: the Ledger is built from
the same (duplicated) transaction rows, so a duplicate's value is already present on both sides of
the comparison.


In [17]:
dup_check = spark.sql('''
    SELECT transaction_id, account_id, posting_date, local_amount,
           MD5(CONCAT_WS('|', account_id, posting_date, transaction_amount, currency,
                          debit_credit_indicator, product_code, gl_account, cost_center)) AS entry_hash
    FROM finance_transactions
''')

from pyspark.sql import Window
from pyspark.sql.functions import row_number, count as spark_count

dup_window = Window.partitionBy("entry_hash", "posting_date").orderBy("transaction_id")
dup_ranked = dup_check.withColumn("rn", row_number().over(dup_window))
dup_group_size = dup_check.groupBy("entry_hash", "posting_date").agg(spark_count("*").alias("group_size"))

duplicate_entries = (
    dup_ranked.join(dup_group_size, ["entry_hash", "posting_date"])
    .filter((col("group_size") > 1) & (col("rn") > 1))
)

duplicate_rows = duplicate_entries.count()
duplicate_amount = duplicate_entries.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
print(f"[INFO] duplicate/re-posted accounting entries: rows={duplicate_rows} amount={duplicate_amount:.2f}")
duplicate_entries.select("transaction_id", "account_id", "posting_date", "local_amount").show(10, truncate=False)


[INFO] duplicate/re-posted accounting entries: rows=21 amount=255845.35


+--------------+-----------+------------+------------+
|transaction_id|account_id |posting_date|local_amount|
+--------------+-----------+------------+------------+
|FTX-DUP001503 |ACC-1000101|2026-08-20  |12574.31    |
|FTX-DUP001519 |ACC-1000336|2026-08-20  |16248.67    |
|FTX-DUP001511 |ACC-1000130|2026-08-17  |3457.66     |
|FTX-DUP001510 |ACC-1000282|2026-08-18  |17099.48    |
|FTX-DUP001517 |ACC-1000378|2026-08-19  |2550.32     |
|FTX-DUP001512 |ACC-1000497|2026-08-20  |16560.67    |
|FTX-DUP001514 |ACC-1000411|2026-08-20  |5873.85     |
|FTX-DUP001509 |ACC-1000441|2026-08-18  |1859.77     |
|FTX-DUP001521 |ACC-1000158|2026-08-18  |5577.10     |
|FTX-DUP001504 |ACC-1000069|2026-08-19  |12533.18    |
+--------------+-----------+------------+------------+
only showing top 10 rows



### Incorrect debit/credit indicator

`ref.accounting_mapping` keys the expected GL account and cost center off *both* the product code
and the debit/credit indicator, so a transaction whose indicator was flipped doesn't just fail to
match its own mapping row - its actual posted classification often becomes an exact match for a
*different*, valid mapping row under the opposite indicator. That's a distinguishable signature: not
some field being wrong, but every field being right, filed under the wrong sign. This is independent
of Task 1's Ledger reconciliation entirely - it reads `bronze.finance_transactions` and
`ref.accounting_mapping` directly, so it isn't affected by that reconciliation's own finding that a
flipped indicator is pass-through to the Ledger's movement figures.


In [18]:
wrong_indicator = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.debit_credit_indicator,
           t.gl_account, t.cost_center, t.local_amount
    FROM finance_transactions t
    JOIN flip_candidates fc ON t.transaction_id = fc.transaction_id
''')
wrong_indicator_count = wrong_indicator.count()
wrong_indicator_amount = wrong_indicator.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
print(f"[INFO] incorrect debit/credit indicator: rows={wrong_indicator_count} amount={wrong_indicator_amount:.2f}")
wrong_indicator.show(10, truncate=False)


[INFO] incorrect debit/credit indicator: rows=9 amount=78480.32


+--------------+------------+----------------------+----------+-----------+------------+
|transaction_id|product_code|debit_credit_indicator|gl_account|cost_center|local_amount|
+--------------+------------+----------------------+----------+-----------+------------+
|FTX-0000124   |P6          |CREDIT                |GL1000    |CC01       |7845.77     |
|FTX-0000660   |P3          |DEBIT                 |GL1002    |CC03       |17900.46    |
|FTX-0000069   |P10         |CREDIT                |GL1003    |CC04       |13435.50    |
|FTX-0000122   |P10         |CREDIT                |GL1003    |CC04       |10615.93    |
|FTX-0001227   |P6          |CREDIT                |GL1000    |CC01       |8050.83     |
|FTX-0001297   |P1          |DEBIT                 |GL1007    |CC08       |11453.83    |
|FTX-0000432   |P6          |CREDIT                |GL1000    |CC01       |7976.36     |
|FTX-0000158   |P8          |DEBIT                 |GL1012    |CC03       |614.95      |
|FTX-0000986   |P7   

### Incorrect FX conversion

`local_amount` is expected to equal `transaction_amount * exchange_rate`, rounded to two decimal
places. Rows more than one minor-currency-unit outside that expectation are flagged. The GL's
local-SGD movement fields are populated from the same transaction-side `local_amount`, so a stale or
wrong FX conversion passes through both sides of the Ledger-vs-source aggregate. This check must read
the transaction's FX inputs directly; the Ledger reconciliation alone cannot distinguish a wrong SGD
amount that both sides share.


In [19]:
from pyspark.sql.functions import round as spark_round

fx_check = txn_df.select(
    "transaction_id", "transaction_amount", "exchange_rate",
    col("local_amount").cast("double").alias("local_amount"),
).withColumn(
    "expected_local_amount", spark_round(col("transaction_amount") * col("exchange_rate"), 2)
).withColumn(
    "fx_variance", col("local_amount") - col("expected_local_amount")
)

fx_mismatches = fx_check.filter(spark_abs(col("fx_variance")) > 0.01)
fx_mismatch_rows = fx_mismatches.count()
fx_mismatch_amount = fx_mismatches.agg(spark_sum("fx_variance")).collect()[0][0] or 0.0
print(f"[INFO] incorrect FX conversion rows={fx_mismatch_rows} amount={fx_mismatch_amount:.2f}")
fx_mismatches.show(10, truncate=False)


[INFO] incorrect FX conversion rows=3 amount=4467.53


+--------------+------------------+-------------+------------+---------------------+------------------+
|transaction_id|transaction_amount|exchange_rate|local_amount|expected_local_amount|fx_variance       |
+--------------+------------------+-------------+------------+---------------------+------------------+
|FTX-0000072   |18900.56          |1.32154324   |27475.7     |24977.91             |2497.790000000001 |
|FTX-0001199   |529.34            |1.34399564   |782.57      |711.43               |71.1400000000001  |
|FTX-0001350   |14159.60          |1.34085776   |20884.61    |18986.01             |1898.6000000000022|
+--------------+------------------+-------------+------------+---------------------+------------------+



### Missing accounting mapping

A transaction with no valid accounting mapping cannot be confirmed against an expected GL account -
reported here as an audit-confidence gap. Excluded from Task 1's expected-classification
recomputation for the same reason: with no expected value to substitute, these transactions keep
their actual classification on both sides of the comparison, so they cannot contribute to the
variance Task 1 finds.


In [20]:
unmapped = spark.sql('''
    SELECT t.transaction_id, t.local_amount
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    )
''')

unmapped_rows = unmapped.count()
unmapped_amount = unmapped.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
print(f"[INFO] missing accounting mapping (audit-confidence gap): rows={unmapped_rows} value={unmapped_amount:.2f}")


[INFO] missing accounting mapping (audit-confidence gap): rows=385 value=4387367.89


### Transaction posted one accounting day late

Candidates are transactions whose posting date is exactly one calendar day after the transaction
date. Each candidate is cross-checked against the per-day movement variance from Task 1 - flagged
only if moving it to the prior day materially improves that prior day's own shortfall. Because the
Ledger's movement figures are aggregated using each transaction's own (possibly late) posting date,
the Ledger and the recomputation already agree on where a late-posted transaction lands on this
dimension - this cross-check is not expected to confirm any candidate; identifying a late posting is
a matter of comparing posting date against transaction date directly (the raw candidate count below).


In [21]:
from pyspark.sql.functions import date_add

late_candidates = txn_df.select(
    "transaction_id", "posting_date", "transaction_date", "legal_entity", "gl_account",
    "cost_center", "currency", "debit_credit_indicator",
    col("local_amount").cast("double").alias("local_amount"),
).filter(col("posting_date") == date_add(col("transaction_date"), 1))

late_candidate_count = late_candidates.count()
print(f"[INFO] posted-one-day-late candidates={late_candidate_count}")

# cross-check: does moving the candidate to transaction_date materially improve that prior day's
# debit/credit variance at the same five-key grain?
prior_day_variance = movement_compare.select(
    col("accounting_date").alias("prior_date"), "legal_entity", "gl_account", "cost_center", "currency",
    "debit_variance", "credit_variance",
)
late_checked = late_candidates.join(
    prior_day_variance,
    (late_candidates.transaction_date == prior_day_variance.prior_date)
    & (late_candidates.legal_entity == prior_day_variance.legal_entity)
    & (late_candidates.gl_account == prior_day_variance.gl_account)
    & (late_candidates.cost_center == prior_day_variance.cost_center)
    & (late_candidates.currency == prior_day_variance.currency),
    "left",
).withColumn(
    "relevant_variance", when(col("debit_credit_indicator") == "DEBIT", col("debit_variance")).otherwise(col("credit_variance"))
)

late_confirmed = late_checked.filter(
    spark_abs(spark_abs(col("relevant_variance")) - col("local_amount")) < 1.0
)
late_confirmed_count = late_confirmed.count()
late_confirmed_amount = late_confirmed.agg(spark_sum("local_amount")).collect()[0][0] or 0.0
print(f"[INFO] posted-one-day-late confirmed against prior-day shortfall: rows={late_confirmed_count} amount={late_confirmed_amount:.2f}")


[INFO] posted-one-day-late candidates=12


[INFO] posted-one-day-late confirmed against prior-day shortfall: rows=0 amount=0.00


### Incorrect legal-entity allocation

The mapping table carries no expected-legal-entity column, so this is a majority-vote check per
account: an account's legal entity is expected to be stable, so a transaction whose legal entity
disagrees with that account's most-frequent posted value elsewhere in the period is a probable
misallocation - the same substitution Task 1's recomputation already applies for this dimension.


In [22]:
account_entity_mode = spark.sql('''
    SELECT account_id, legal_entity, COUNT(*) AS n,
           ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY COUNT(*) DESC) AS rnk
    FROM finance_transactions
    GROUP BY account_id, legal_entity
''').filter(col("rnk") == 1)

wrong_entity = txn_df.alias("t").join(
    account_entity_mode.alias("e"), col("t.account_id") == col("e.account_id")
).filter(col("t.legal_entity") != col("e.legal_entity")).select(
    col("t.transaction_id"), col("t.legal_entity").alias("actual_entity"), col("e.legal_entity").alias("expected_entity"),
    col("t.local_amount").cast("double").alias("local_amount"),
)

wrong_entity_rows = wrong_entity.count()
wrong_entity_amount = wrong_entity.agg(spark_sum("local_amount")).collect()[0][0] or 0.0
print(f"[INFO] incorrect legal-entity allocation: rows={wrong_entity_rows} amount={wrong_entity_amount:.2f}")
wrong_entity.show(10, truncate=False)


[INFO] incorrect legal-entity allocation: rows=6 amount=60697.50


+--------------+-------------+---------------+------------+
|transaction_id|actual_entity|expected_entity|local_amount|
+--------------+-------------+---------------+------------+
|FTX-0001392   |LE1          |LE3            |18465.48    |
|FTX-0001387   |LE1          |LE2            |1323.82     |
|FTX-0000775   |LE1          |LE2            |15558.97    |
|FTX-0000896   |LE3          |LE4            |12251.32    |
|FTX-0000706   |LE3          |LE1            |120.03      |
|FTX-0000519   |LE2          |LE4            |12977.88    |
+--------------+-------------+---------------+------------+



### Incorrect GL-account and cost-center assignment

The mapping table's expected GL account and expected cost center come from the same effective-dated
join, restricted to a transaction's single, currently-active, unambiguous mapping match (a
product/transaction-type combination with more than one currently-active, conflicting row has no
single expected value to check against, so it is excluded rather than checked against an arbitrary
pick among its candidates - the same restriction Task 1's recomputation applies). GL account and cost
center diverge independently from that one match: a transaction can be wrong on GL account alone,
cost center alone, or both at once, so the three shapes are reported as three separate populations
rather than folded into one count.

In [23]:
classification_movers = spark.sql('''
    SELECT t.transaction_id, t.gl_account AS actual_gl_account, cm.expected_gl_account,
           t.cost_center AS actual_cost_center, cm.expected_cost_center, t.local_amount
    FROM finance_transactions t
    JOIN canonical_mapping cm ON t.transaction_id = cm.transaction_id
    WHERE t.gl_account <> cm.expected_gl_account OR t.cost_center <> cm.expected_cost_center
''')
classification_movers.cache()

wrong_gl_account = classification_movers.filter(
    (col("actual_gl_account") != col("expected_gl_account")) & (col("actual_cost_center") == col("expected_cost_center"))
)
wrong_cost_center = classification_movers.filter(
    (col("actual_gl_account") == col("expected_gl_account")) & (col("actual_cost_center") != col("expected_cost_center"))
)
wrong_gl_and_cost_center = classification_movers.filter(
    (col("actual_gl_account") != col("expected_gl_account")) & (col("actual_cost_center") != col("expected_cost_center"))
)

wrong_gl_rows = wrong_gl_account.count()
wrong_gl_amount = wrong_gl_account.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
wrong_cc_txns = wrong_cost_center.count()
wrong_cc_amount = wrong_cost_center.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
wrong_both_rows = wrong_gl_and_cost_center.count()
wrong_both_amount = wrong_gl_and_cost_center.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0

print(f"[INFO] incorrect GL-account assignment: rows={wrong_gl_rows} amount={wrong_gl_amount:.2f}")
print(f"[INFO] incorrect cost-center assignment: rows={wrong_cc_txns} amount={wrong_cc_amount:.2f}")
print(f"[INFO] incorrect GL-account and cost-center assignment (same transaction): rows={wrong_both_rows} amount={wrong_both_amount:.2f}")
classification_movers.show(10, truncate=False)


[INFO] incorrect GL-account assignment: rows=6 amount=53687.44
[INFO] incorrect cost-center assignment: rows=3 amount=40036.00
[INFO] incorrect GL-account and cost-center assignment (same transaction): rows=1 amount=9706.73


+--------------+-----------------+-------------------+------------------+--------------------+------------+
|transaction_id|actual_gl_account|expected_gl_account|actual_cost_center|expected_cost_center|local_amount|
+--------------+-----------------+-------------------+------------------+--------------------+------------+
|FTX-0000068   |GL1011           |GL1011             |CC09              |CC02                |16506.33    |
|FTX-0000142   |GL1007           |GL1011             |CC02              |CC02                |6339.34     |
|FTX-0000173   |GL1007           |GL1011             |CC02              |CC02                |11891.61    |
|FTX-0000894   |GL1005           |GL1012             |CC03              |CC03                |16043.43    |
|FTX-0001039   |GL1005           |GL1012             |CC03              |CC03                |16124.61    |
|FTX-0001171   |GL1005           |GL1012             |CC03              |CC03                |1477.99     |
|FTX-0001243   |GL1007      

### Findings summary and Ledger reconciliation

Task 1's expected-classification recomputation is the top-down figure the categories below are
checked against - not the write-back `amount` dimension (a grand total, unaffected by which bucket a
transaction's value lands in), but the sum of absolute per-key variances from the dimensional
recomputation, which does move when a transaction is classified differently than the Ledger recorded
it. The categories that dimension can move on - legal-entity, GL-account, and cost-center
misclassification - are checked directly against it below, rather than estimated: every transaction
carrying a different expected value than its posted one is reverted to its actual, as-posted
classification and the recomputation is re-run, so the bridge is confirmed rather than assumed.

In [24]:
findings = [
    ("duplicate / re-posted accounting entries", duplicate_rows, duplicate_amount),
    ("incorrect debit/credit indicator", wrong_indicator_count, wrong_indicator_amount),
    ("incorrect FX conversion", fx_mismatch_rows, fx_mismatch_amount),
    ("missing accounting mapping", unmapped_rows, unmapped_amount),
    ("posted one accounting day late (confirmed)", late_confirmed_count, late_confirmed_amount),
    ("incorrect legal-entity allocation", wrong_entity_rows, wrong_entity_amount),
    ("incorrect GL-account assignment", wrong_gl_rows, wrong_gl_amount),
    ("incorrect cost-center assignment", wrong_cc_txns, wrong_cc_amount),
    ("incorrect GL-account and cost-center assignment (same transaction)", wrong_both_rows, wrong_both_amount),
]

print("[INFO] category findings:")
for name, rows, amount in findings:
    amount_str = f"{amount:.2f}" if amount is not None else "n/a"
    print(f"  {name}: rows={rows} SGD amount={amount_str}")

print(f"[INFO] independent Ledger SGD variance (expected-classification basis, from task 1): {movement_variance_total:.2f}")


[INFO] category findings:
  duplicate / re-posted accounting entries: rows=21 SGD amount=255845.35
  incorrect debit/credit indicator: rows=9 SGD amount=78480.32
  incorrect FX conversion: rows=3 SGD amount=4467.53
  missing accounting mapping: rows=385 SGD amount=4387367.89
  posted one accounting day late (confirmed): rows=0 SGD amount=0.00
  incorrect legal-entity allocation: rows=6 SGD amount=60697.50
  incorrect GL-account assignment: rows=6 SGD amount=53687.44
  incorrect cost-center assignment: rows=3 SGD amount=40036.00
  incorrect GL-account and cost-center assignment (same transaction): rows=1 SGD amount=9706.73
[INFO] independent Ledger SGD variance (expected-classification basis, from task 1): 297137.40


### Closing the bridge to Task 1's recomputation

Every transaction Task 1's recomputation substitutes a different classification for is, by
construction, the complete and only source of the movement variance it finds: a transaction whose
actual and expected classification already agree contributes identically to the Ledger and to the
recomputation, on every dimension, so it cannot be a source of variance either way. The bridge is
confirmed directly rather than by summing each category's face value and doubling it - reverting
exactly the flagged transactions to their actual, as-posted classification and re-running the
recomputation should return zero variance on every key if the categories above are the complete
explanation.

In [25]:
known_movers = (
    wrong_entity.select("transaction_id")
    .unionByName(classification_movers.select("transaction_id"))
    .distinct()
)
known_movers.createOrReplaceTempView("known_movers")

txn_reverted = txn_df.alias("t").join(
    canonical_mapping.alias("cm"), col("t.transaction_id") == col("cm.transaction_id"), "left"
).join(
    account_entity_mode_t1.alias("em"), col("t.account_id") == col("em.account_id"), "left"
).join(
    known_movers.alias("km"), col("t.transaction_id") == col("km.transaction_id"), "left"
).select(
    col("t.posting_date").alias("accounting_date"),
    when(col("km.transaction_id").isNotNull(), col("t.legal_entity"))
        .otherwise(coalesce(col("em.legal_entity"), col("t.legal_entity"))).alias("legal_entity"),
    when(col("km.transaction_id").isNotNull(), col("t.gl_account"))
        .otherwise(coalesce(col("cm.expected_gl_account"), col("t.gl_account"))).alias("gl_account"),
    when(col("km.transaction_id").isNotNull(), col("t.cost_center"))
        .otherwise(coalesce(col("cm.expected_cost_center"), col("t.cost_center"))).alias("cost_center"),
    col("t.currency"), col("t.debit_credit_indicator"),
    col("t.local_amount").cast("double").alias("recompute_amount"),
)
recomputed_reverted = txn_reverted.groupBy(
    "accounting_date", "legal_entity", "gl_account", "cost_center", "currency"
).agg(
    spark_sum(when(col("debit_credit_indicator") == "DEBIT", col("recompute_amount")).otherwise(0.0)).alias("recomputed_debit"),
    spark_sum(when(col("debit_credit_indicator") == "CREDIT", col("recompute_amount")).otherwise(0.0)).alias("recomputed_credit"),
)
bridge_compare = gl.join(recomputed_reverted, JOIN_KEYS, "full_outer").fillna(
    0.0, subset=["debit_movement", "recomputed_debit", "credit_movement", "recomputed_credit"]
).withColumn("debit_variance", col("debit_movement") - col("recomputed_debit")
).withColumn("credit_variance", col("credit_movement") - col("recomputed_credit"))

bridge_fail_keys = bridge_compare.filter(
    (spark_abs(col("debit_variance")) > 0.01) | (spark_abs(col("credit_variance")) > 0.01)
)
bridge_residual = bridge_fail_keys.agg(
    spark_sum(spark_abs(col("debit_variance")) + spark_abs(col("credit_variance")))
).collect()[0][0] or 0.0

print(f"[INFO] transactions bridged: {known_movers.count()}")
print(f"[INFO] keys exceeding tolerance once bridged transactions are reverted to actual: {bridge_fail_keys.count()}")
print(f"[INFO] residual SGD movement variance after the bridge: {bridge_residual:.2f}")


[INFO] transactions bridged: 16


[INFO] keys exceeding tolerance once bridged transactions are reverted to actual: 0
[INFO] residual SGD movement variance after the bridge: 0.00


In [26]:
print("[PASS] task 3 variance investigation: all named categories checked")
spark.stop()


[PASS] task 3 variance investigation: all named categories checked


## Exception Dataset

One row per flagged transaction/issue-type pair, minimum columns: transaction id, issue type, source
value, comparison value, variance. Combines the accounting-mapping-validation findings (Task 2) and
the variance-investigation findings (Task 3) into a single dataset.


In [27]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-exception-dataset")
    .getOrCreate()
)

txn_df = jdbc_table("bronze.finance_transactions")
map_df = jdbc_table("ref.accounting_mapping")
gl_df = jdbc_table("finance.gl_balance")
txn_df.createOrReplaceTempView("finance_transactions")
map_df.createOrReplaceTempView("accounting_mapping")

# a transaction whose actual (gl_account, cost_center) doesn't match any active mapping row for its
# own posted indicator, but exactly matches one for the opposite indicator - see task 3's own check.
# registered first so the GL-account/cost-center checks below can look a flipped-indicator
# transaction up under its corrected indicator rather than its own (wrong) posted one.
flip_candidates_ds = spark.sql('''
    WITH current_match AS (
      SELECT t.transaction_id,
             MAX(CASE WHEN t.gl_account = m.expected_gl_account AND t.cost_center = m.expected_cost_center THEN 1 ELSE 0 END) AS matches_current
      FROM finance_transactions t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
        AND t.transaction_date >= m.effective_start_date AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
      GROUP BY t.transaction_id
    ),
    opposite_match AS (
      SELECT t.transaction_id,
             MAX(CASE WHEN t.gl_account = m.expected_gl_account AND t.cost_center = m.expected_cost_center THEN 1 ELSE 0 END) AS matches_opposite
      FROM finance_transactions t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code
        AND m.transaction_type = CASE WHEN t.debit_credit_indicator = 'DEBIT' THEN 'CREDIT' ELSE 'DEBIT' END
        AND t.transaction_date >= m.effective_start_date AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
      GROUP BY t.transaction_id
    )
    SELECT t.transaction_id
    FROM finance_transactions t
    LEFT JOIN current_match cm ON t.transaction_id = cm.transaction_id
    JOIN opposite_match om ON t.transaction_id = om.transaction_id
    WHERE COALESCE(cm.matches_current, 0) = 0 AND om.matches_opposite = 1
''')
flip_candidates_ds.createOrReplaceTempView("flip_candidates_ds")

# the mapping lookup itself is keyed on the transaction's indicator - a flip candidate's own posted
# indicator is the wrong key to look up under, so the opposite indicator is used instead, exactly as
# task 1/task 3's own canonical_mapping does.
canonical_mapping_ds = spark.sql('''
    WITH corrected AS (
      SELECT t.*, CASE WHEN fc.transaction_id IS NOT NULL
                        THEN (CASE WHEN t.debit_credit_indicator = 'DEBIT' THEN 'CREDIT' ELSE 'DEBIT' END)
                        ELSE t.debit_credit_indicator END AS lookup_type
      FROM finance_transactions t
      LEFT JOIN flip_candidates_ds fc ON t.transaction_id = fc.transaction_id
    ),
    matched AS (
      SELECT t.transaction_id, m.expected_gl_account, m.expected_cost_center,
             COUNT(*) OVER (PARTITION BY t.transaction_id) AS match_count
      FROM corrected t
      JOIN accounting_mapping m
        ON t.product_code = m.product_code AND t.lookup_type = m.transaction_type
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    )
    SELECT DISTINCT transaction_id, expected_gl_account, expected_cost_center
    FROM matched WHERE match_count = 1
''')
canonical_mapping_ds.createOrReplaceTempView("canonical_mapping_ds")

mapping_exceptions_ds = spark.sql('''
    SELECT t.transaction_id, 'GL_MISMATCH' AS issue_type,
           t.gl_account AS source_value, m.expected_gl_account AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON  t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
      AND t.transaction_date >= m.effective_start_date
      AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    WHERE t.gl_account <> m.expected_gl_account

    UNION ALL

    SELECT t.transaction_id, 'NO_EFFECTIVE_MAPPING' AS issue_type,
           t.gl_account AS source_value, CAST(NULL AS STRING) AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    WHERE EXISTS (
      SELECT 1 FROM accounting_mapping m WHERE m.product_code = t.product_code
        AND m.transaction_type = t.debit_credit_indicator)
    AND NOT EXISTS (
      SELECT 1 FROM accounting_mapping m WHERE m.product_code = t.product_code
        AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL))

    UNION ALL

    SELECT t.transaction_id, 'MAPPING_NOT_FOUND' AS issue_type,
           t.gl_account AS source_value, CAST(NULL AS STRING) AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m WHERE m.product_code = t.product_code
        AND m.transaction_type = t.debit_credit_indicator)

    UNION ALL

    SELECT t.transaction_id, 'EXPIRED_MAPPING' AS issue_type,
           t.gl_account AS source_value, m.expected_gl_account AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
    WHERE m.effective_end_date IS NOT NULL
      AND t.transaction_date > m.effective_end_date
      AND NOT EXISTS (
        SELECT 1 FROM accounting_mapping m2
        WHERE m2.product_code = t.product_code AND m2.transaction_type = t.debit_credit_indicator
          AND t.transaction_date >= m2.effective_start_date
          AND (t.transaction_date <= m2.effective_end_date OR m2.effective_end_date IS NULL))

    UNION ALL

    SELECT t.transaction_id, 'WRONG_GL_ACCOUNT' AS issue_type,
           t.gl_account AS source_value, cm.expected_gl_account AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN canonical_mapping_ds cm ON t.transaction_id = cm.transaction_id
    WHERE t.gl_account <> cm.expected_gl_account

    UNION ALL

    SELECT t.transaction_id, 'WRONG_COST_CENTER' AS issue_type,
           t.cost_center AS source_value, cm.expected_cost_center AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN canonical_mapping_ds cm ON t.transaction_id = cm.transaction_id
    WHERE t.cost_center <> cm.expected_cost_center

    UNION ALL

    SELECT t.transaction_id, 'UNMAPPED_VARIANCE' AS issue_type,
           CAST(t.local_amount AS STRING) AS source_value, CAST(NULL AS STRING) AS comparison_value,
           t.local_amount AS variance
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL))
''')

fx_exceptions_ds = spark.sql('''
    SELECT transaction_id, 'FX_CONVERSION_ERROR' AS issue_type,
           CAST(local_amount AS STRING) AS source_value,
           CAST(ROUND(transaction_amount * exchange_rate, 2) AS STRING) AS comparison_value,
           local_amount - ROUND(transaction_amount * exchange_rate, 2) AS variance
    FROM finance_transactions
    WHERE ABS(local_amount - ROUND(transaction_amount * exchange_rate, 2)) > 0.01
''')

duplicate_exceptions_ds = spark.sql('''
    WITH hashed AS (
      SELECT transaction_id, account_id, posting_date, local_amount,
             MD5(CONCAT_WS('|', account_id, posting_date, transaction_amount, currency,
                            debit_credit_indicator, product_code, gl_account, cost_center)) AS entry_hash
      FROM finance_transactions
    ),
    ranked AS (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY entry_hash, posting_date ORDER BY transaction_id) AS rn,
             COUNT(*) OVER (PARTITION BY entry_hash, posting_date) AS group_size
      FROM hashed
    )
    SELECT transaction_id, 'DUPLICATE_ENTRY' AS issue_type,
           CAST(local_amount AS STRING) AS source_value, CAST(NULL AS STRING) AS comparison_value,
           local_amount AS variance
    FROM ranked WHERE group_size > 1 AND rn > 1
''')

wrong_entity_exceptions_ds = spark.sql('''
    WITH account_entity_mode AS (
      SELECT account_id, legal_entity,
             ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY COUNT(*) DESC) AS rnk
      FROM finance_transactions
      GROUP BY account_id, legal_entity
    )
    SELECT t.transaction_id, 'WRONG_LEGAL_ENTITY' AS issue_type,
           t.legal_entity AS source_value, e.legal_entity AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN account_entity_mode e ON t.account_id = e.account_id AND e.rnk = 1
    WHERE t.legal_entity <> e.legal_entity
''')

wrong_indicator_exceptions_ds = spark.sql('''
    SELECT t.transaction_id, 'WRONG_DR_CR_INDICATOR' AS issue_type,
           t.debit_credit_indicator AS source_value, CAST(NULL AS STRING) AS comparison_value,
           t.local_amount AS variance
    FROM finance_transactions t
    JOIN flip_candidates_ds fc ON t.transaction_id = fc.transaction_id
''')

exception_dataset = (
    mapping_exceptions_ds
    .unionByName(fx_exceptions_ds)
    .unionByName(duplicate_exceptions_ds)
    .unionByName(wrong_entity_exceptions_ds)
    .unionByName(wrong_indicator_exceptions_ds)
)
exception_dataset.cache()
total_exception_rows = exception_dataset.count()
print(f"[INFO] exception dataset rows={total_exception_rows}")
exception_dataset.groupBy("issue_type").count().orderBy(col("count").desc()).show(truncate=False)
exception_dataset.show(10, truncate=False)


[INFO] exception dataset rows=1223


+---------------------+-----+
|issue_type           |count|
+---------------------+-----+
|GL_MISMATCH          |403  |
|MAPPING_NOT_FOUND    |385  |
|UNMAPPED_VARIANCE    |385  |
|DUPLICATE_ENTRY      |21   |
|WRONG_DR_CR_INDICATOR|9    |
|WRONG_GL_ACCOUNT     |7    |
|WRONG_LEGAL_ENTITY   |6    |
|WRONG_COST_CENTER    |4    |
|FX_CONVERSION_ERROR  |3    |
+---------------------+-----+



+--------------+-----------+------------+----------------+--------+
|transaction_id|issue_type |source_value|comparison_value|variance|
+--------------+-----------+------------+----------------+--------+
|FTX-0000008   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000012   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000020   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000023   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000027   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000042   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000080   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000100   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000114   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000156   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
+--------------+-----------+------------+----------------+--------+
only showing top 10 rows



In [28]:
print(f"[PASS] exception dataset: rows={total_exception_rows}")
spark.stop()


[PASS] exception dataset: rows=1223
